# 🤖 Responses API 互換サーバー — Qwen2.5 1.5B + Function Calling (CPU)

| 項目 | 内容 |
|---|---|
| バックエンド | llama-cpp-python (CPU) |
| モデル | Qwen2.5-1.5B-Instruct (Q4_K_M GGUF) — 認証不要 |
| 機能 | テキスト生成 / Tool call (Function calling) |
| 公開 | cloudflared によるパブリックURL (アカウント不要) |

## エンドポイント一覧
| エンドポイント | 説明 |
|---|---|
| `POST /v1/responses` | Responses API（ツール呼び出し対応） |
| `GET /v1/models` | モデル一覧 |
| `GET /health` | ヘルスチェック |

## Step 1: 依存ライブラリのインストール

In [ ]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu -q
!pip install fastapi uvicorn[standard] huggingface_hub -q
print('✅ インストール完了')

## Step 2: Qwen2.5-1.5B (GGUF) のダウンロード

In [ ]:
from huggingface_hub import hf_hub_download
import os

# ---- モデル設定 ----
# ✅ 認証不要・制限なし・Function calling対応
# 他の候補:
#   Qwen2.5-3B  : REPO="Qwen/Qwen2.5-3B-Instruct-GGUF"  FILE="qwen2.5-3b-instruct-q4_k_m.gguf"  (~2.0GB)
#   Qwen2.5-0.5B: REPO="Qwen/Qwen2.5-0.5B-Instruct-GGUF" FILE="qwen2.5-0.5b-instruct-q4_k_m.gguf" (~400MB)
REPO_ID   = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
FILENAME  = "qwen2.5-1.5b-instruct-q4_k_m.gguf"
MODEL_DIR = "/content/models"
# --------------------

os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, FILENAME)

if not os.path.exists(MODEL_PATH):
    print(f'📥 ダウンロード中: {FILENAME} (~1.1GB)...')
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=MODEL_DIR)
    print('✅ ダウンロード完了')
else:
    print(f'✅ キャッシュ済み: {MODEL_PATH}')

MODEL_NAME = FILENAME

## Step 3: Responses API サーバーの定義

In [ ]:
server_code = r'''
import os, re, time, uuid, json
from typing import Optional, List, Any, Union
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from llama_cpp import Llama

# ===== モデルロード =====
MODEL_PATH = os.environ["MODEL_PATH"]
N_CTX      = int(os.environ.get("N_CTX", "4096"))
N_THREADS  = int(os.environ.get("N_THREADS", str(os.cpu_count() or 2)))

print(f"Loading: {MODEL_PATH}  ctx={N_CTX}  threads={N_THREADS}")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    verbose=False,
)
print("Model ready.")

MODEL_NAME = os.path.basename(MODEL_PATH)
app = FastAPI(title="Responses API (Qwen2.5 + Tools)", version="1.0.0")

# ===== スキーマ =====
class FunctionDef(BaseModel):
    name: str
    description: Optional[str] = ""
    parameters: Optional[dict] = {}
    strict: Optional[bool] = False

class ToolDef(BaseModel):
    type: str = "function"
    name: Optional[str] = None
    description: Optional[str] = None
    parameters: Optional[dict] = None
    function: Optional[FunctionDef] = None

class MessageItem(BaseModel):
    role: str
    content: Union[str, list, None] = None

class ResponsesRequest(BaseModel):
    model: str
    input: Union[str, List[MessageItem]]
    instructions: Optional[str] = None
    tools: Optional[List[ToolDef]] = None
    tool_choice: Optional[Any] = "auto"
    max_output_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.95

# ===== ユーティリティ =====
def normalize_tools(tools: List[ToolDef]) -> list:
    result = []
    for t in tools:
        if t.type == "function":
            if t.function:
                fn = {"name": t.function.name, "description": t.function.description,
                      "parameters": t.function.parameters or {}}
            else:
                fn = {"name": t.name or "", "description": t.description or "",
                      "parameters": t.parameters or {}}
            result.append({"type": "function", "function": fn})
    return result

def build_messages(req: ResponsesRequest) -> list:
    msgs = [{"role": "system", "content": req.instructions or "You are a helpful assistant."}]
    if isinstance(req.input, str):
        msgs.append({"role": "user", "content": req.input})
    else:
        for m in req.input:
            if isinstance(m.content, list):
                text = " ".join(c.get("text", "") if isinstance(c, dict) else getattr(c, "text", "") for c in m.content)
            else:
                text = m.content or ""
            msgs.append({"role": m.role, "content": text})
    return msgs

def make_text_output(response_id, model, text, usage, created_at):
    return {
        "id": response_id, "object": "response", "created_at": created_at, "model": model,
        "output": [{
            "type": "message",
            "id": f"msg_{uuid.uuid4().hex[:12]}",
            "role": "assistant",
            "content": [{"type": "output_text", "text": text}],
            "status": "completed",
        }],
        "usage": usage, "status": "completed",
    }

def make_tool_output(response_id, model, tool_calls, usage, created_at):
    output_items = []
    for tc in tool_calls:
        fn = tc.get("function", {})
        raw_args = fn.get("arguments", "{}")
        try:
            args = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
        except Exception:
            args = {}
        call_id = tc.get("id", f"call_{uuid.uuid4().hex[:12]}")
        output_items.append({
            "type": "function_call",
            "id": call_id,
            "call_id": call_id,
            "name": fn.get("name", ""),
            "arguments": json.dumps(args, ensure_ascii=False),
            "status": "completed",
        })
    return {
        "id": response_id, "object": "response", "created_at": created_at, "model": model,
        "output": output_items,
        "usage": usage, "status": "completed",
    }

# ===== エンドポイント =====
@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [
        {"id": MODEL_NAME, "object": "model", "created": int(time.time()), "owned_by": "local"}
    ]}

@app.post("/v1/responses")
async def create_response(req: ResponsesRequest):
    messages    = build_messages(req)
    response_id = f"resp_{uuid.uuid4().hex[:16]}"
    created_at  = int(time.time())
    kwargs = dict(
        messages=messages,
        max_tokens=req.max_output_tokens,
        temperature=req.temperature,
        top_p=req.top_p,
    )

    if req.tools:
        lc_tools = normalize_tools(req.tools)
        tc_choice = "auto"
        if isinstance(req.tool_choice, str):
            tc_choice = req.tool_choice if req.tool_choice in ("auto", "none") else "auto"
        elif isinstance(req.tool_choice, dict):
            tc_choice = req.tool_choice
        kwargs["tools"] = lc_tools
        kwargs["tool_choice"] = tc_choice

    result  = llm.create_chat_completion(**kwargs)
    choice  = result["choices"][0]
    usage   = {
        "input_tokens":  result["usage"]["prompt_tokens"],
        "output_tokens": result["usage"]["completion_tokens"],
        "total_tokens":  result["usage"]["total_tokens"],
    }

    tool_calls = choice["message"].get("tool_calls") or []

    # Qwen系モデルは tool_calls の代わりにテキスト内へ
    # <tool_call>{...}</tool_call> を出力する場合があるためフォールバックでパース
    if not tool_calls:
        text = choice["message"].get("content") or ""
        for raw in re.findall(r"<tool_call>\s*(\{+.*?\}+)\s*</tool_call>", text, re.DOTALL):
            try:
                # Qwen が {{...}} と二重波括弧で出力する場合があるため正規化
                normalized = re.sub(r"\{\{", "{", re.sub(r"\}\}", "}", raw))
                obj = json.loads(normalized)
                tool_calls.append({
                    "id": f"call_{uuid.uuid4().hex[:12]}",
                    "function": {
                        "name": obj.get("name", ""),
                        "arguments": json.dumps(obj.get("arguments", {}), ensure_ascii=False),
                    }
                })
            except Exception:
                pass

    if tool_calls:
        return JSONResponse(make_tool_output(response_id, req.model, tool_calls, usage, created_at))
    else:
        text = choice["message"].get("content") or ""
        return JSONResponse(make_text_output(response_id, req.model, text, usage, created_at))
'''

with open('/content/responses_api_server.py', 'w') as f:
    f.write(server_code)

print('✅ サーバーコード書き出し完了')

## Step 4: cloudflared のインストール

In [ ]:
# cloudflared インストール (ngrok不要・アカウント不要)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ cloudflared インストール完了')

## Step 5: サーバー起動 🚀

In [ ]:
import subprocess, threading, time, os, re

PORT = 8000
env = os.environ.copy()
env["MODEL_PATH"] = MODEL_PATH

# 旧プロセスを確実に終了
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(2)

# uvicorn 起動
proc = subprocess.Popen(
    ["uvicorn", "responses_api_server:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd="/content", env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
threading.Thread(target=lambda: [print(l.decode().rstrip()) for l in proc.stdout], daemon=True).start()

print('⏳ モデルロード中 (30秒ほどかかります)...')
time.sleep(30)

# cloudflared トンネル起動
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(60):
    line = cf_proc.stdout.readline().decode()
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        break

print()
print('=' * 60)
print(f'🌐 パブリックURL : {PUBLIC_URL}')
print(f'📋 Responses API: {PUBLIC_URL}/v1/responses')
print(f'📋 ヘルスチェック: {PUBLIC_URL}/health')
print('=' * 60)

## Step 6: 動作確認 — テキスト生成

In [ ]:
import requests, json

# ヘルスチェック
r = requests.get(f"{PUBLIC_URL}/health")
print("Health:", r.json())

# テキスト生成
payload = {
    "model": MODEL_NAME,
    "input": "Pythonでフィボナッチ数列を返す関数を書いてください。",
    "instructions": "あなたは優秀なプログラミングアシスタントです。",
    "max_output_tokens": 400,
    "temperature": 0.5,
}

r = requests.post(f"{PUBLIC_URL}/v1/responses", json=payload)
data = r.json()
print("\n--- 出力 ---")
print(data["output"][0]["content"][0]["text"])
print("Usage:", data["usage"])

## Step 7: 動作確認 — Function Calling

In [ ]:
import requests, json

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "指定した都市の現在の天気情報を取得する",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "都市名 (例: Tokyo, Osaka)"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city"]
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "数式を計算する",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "計算する数式 (例: 2 + 3 * 4)"}
            },
            "required": ["expression"]
        }
    }
]

# 単発ツール呼び出し
payload = {
    "model": MODEL_NAME,
    "input": "東京の天気を摂氏で教えてください。",
    "tools": tools,
    "tool_choice": "auto",
    "max_output_tokens": 256,
    "temperature": 0.1,
}

r = requests.post(f"{PUBLIC_URL}/v1/responses", json=payload)
data = r.json()

print("--- レスポンス ---")
for item in data["output"]:
    if item["type"] == "function_call":
        args = json.loads(item["arguments"])
        print(f"🔧 ツール呼び出し: {item['name']}")
        print(f"   引数: {json.dumps(args, ensure_ascii=False, indent=4)}")
    elif item["type"] == "message":
        print(f"💬 テキスト: {item['content'][0]['text']}")
print("Usage:", data["usage"])

In [ ]:
import requests, json

# マルチターン: ツール結果を返して最終回答を得る
payload1 = {
    "model": MODEL_NAME,
    "input": "(123 + 456) * 7 を計算してください。式はそのまま expression に入れてください。",
    "tools": tools,
    "tool_choice": "auto",
    "max_output_tokens": 128,
    "temperature": 0.1,
}
r1 = requests.post(f"{PUBLIC_URL}/v1/responses", json=payload1).json()
fc = r1["output"][0]
print(f"1️⃣ ツール呼び出し: {fc['name']}({fc['arguments']})")

expr = json.loads(fc["arguments"])["expression"]
try:
    result_value = eval(expr)
except Exception:
    result_value = "計算できませんでした"
print(f"2️⃣ 計算結果: {expr} = {result_value}")

payload2 = {
    "model": MODEL_NAME,
    "input": [
        {"role": "user",      "content": "(123 + 456) * 7 を計算してください。"},
        {"role": "assistant", "content": f"[tool_call: {fc['name']} => {result_value}]"},
        {"role": "user",      "content": f"ツール結果: {result_value}。この結果を使って答えてください。"},
    ],
    "max_output_tokens": 256,
    "temperature": 0.3,
}
r2 = requests.post(f"{PUBLIC_URL}/v1/responses", json=payload2).json()
print(f"3️⃣ 最終回答: {r2['output'][0]['content'][0]['text']}")

## Step 8: 独自ツールの定義とエージェントループ

In [ ]:
import requests, json

MY_TOOLS = [
    {
        "type": "function",
        "name": "search_database",
        "description": "社内データベースを検索する。検索キーワードは必ず query パラメータに入れること。",
        "parameters": {
            "type": "object",
            "properties": {
                "query":  {"type": "string",  "description": "検索キーワード (必須)"},
                "limit":  {"type": "integer", "description": "取得件数 (デフォルト: 5)"},
                "filter": {"type": "string",  "description": "絞り込み条件 (任意)"}
            },
            "required": ["query"]
        }
    }
]

def execute_tool(name: str, arguments: dict):
    """ツールを実際に実行する — ここに本物の処理を実装してください"""
    if name == "search_database":
        # query が空の場合は filter にフォールバック
        keyword = arguments.get("query") or arguments.get("filter") or ""
        return {
            "results": [f"ダミー結果: {keyword} に関するドキュメント{i+1}" for i in range(3)],
            "count": 3
        }
    return {"error": f"未知のツール: {name}"}

def agent_loop(user_input: str, max_turns: int = 5):
    """ツールが不要になるまで自動でループするエージェント"""
    history = [{"role": "user", "content": user_input}]

    for turn in range(max_turns):
        payload = {
            "model": MODEL_NAME,
            "input": history,
            "tools": MY_TOOLS,
            "tool_choice": "auto",
            "max_output_tokens": 512,
            "temperature": 0.2,
        }
        resp = requests.post(f"{PUBLIC_URL}/v1/responses", json=payload).json()
        output = resp["output"]

        tool_calls = [o for o in output if o["type"] == "function_call"]
        if not tool_calls:
            return next((o["content"][0]["text"] for o in output if o["type"] == "message"), "")

        for tc in tool_calls:
            args = json.loads(tc["arguments"])
            result = execute_tool(tc["name"], args)
            print(f"  🔧 {tc['name']}({args}) => {result}")
            history.append({"role": "assistant", "content": f"[tool_call:{tc['name']}]"})
            history.append({"role": "user",      "content": f"ツール結果: {json.dumps(result, ensure_ascii=False)}"})

    return "(最大ターン数に達しました)"

print("=== エージェントループ テスト ===")
answer = agent_loop("'機械学習' について社内データベースを検索して、結果を要約してください。")
print(f"\n最終回答:\n{answer}")

## Step 9: サーバー停止

In [ ]:
import subprocess
cf_proc.terminate()
proc.terminate()
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
print('🛑 サーバー停止完了')

## 📝 Tips

### Function calling のコツ
- `temperature` は **0.1〜0.2** に下げると安定します
- `description` に「`query` パラメータに必ずキーワードを入れること」と明示すると精度が上がります
- ツール定義は **3つ以下** を推奨（小型モデルの限界）

### 他モデルへの切り替え
Step 2 の2行を変えるだけです:
```python
# 精度重視 (3B)
REPO_ID  = "Qwen/Qwen2.5-3B-Instruct-GGUF"
FILENAME = "qwen2.5-3b-instruct-q4_k_m.gguf"

# 速度重視 (0.5B)
REPO_ID  = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
```

### Colab CPU スペック目安
| モデル | 推論速度 | メモリ使用量 |
|---|---|---|
| Qwen2.5-0.5B Q4_K_M | ~20 tok/s | ~0.5 GB |
| Qwen2.5-1.5B Q4_K_M | ~10 tok/s | ~1.1 GB |
| Qwen2.5-3B Q4_K_M   | ~5 tok/s  | ~2.5 GB |